<a href="https://colab.research.google.com/github/msedawy045-hub/ifikra-task/blob/main/ifikra_user_domain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install neo4j chromadb sentence-transformers

In [19]:
from google.colab import userdata
from neo4j import GraphDatabase

uri = userdata.get("NEO4J_URI")
user = userdata.get("NEO4J_USERNAME")
pwd = userdata.get("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(user, pwd))
driver.verify_connectivity()
print("Connected to Neo4j ✅")

Connected to Neo4j ✅


In [20]:
def run(query, **params):
    with driver.session() as session:
        return session.run(query, **params).data()

constraints = [
    "CREATE CONSTRAINT user_id IF NOT EXISTS FOR (u:User) REQUIRE u.id IS UNIQUE",
    "CREATE CONSTRAINT enterprise_id IF NOT EXISTS FOR (e:Enterprise) REQUIRE e.id IS UNIQUE",
    "CREATE CONSTRAINT company_id IF NOT EXISTS FOR (c:Company) REQUIRE c.id IS UNIQUE",
    "CREATE CONSTRAINT team_id IF NOT EXISTS FOR (t:Team) REQUIRE t.id IS UNIQUE",
    "CREATE CONSTRAINT role_name IF NOT EXISTS FOR (r:Role) REQUIRE r.name IS UNIQUE",
    "CREATE CONSTRAINT actor_name IF NOT EXISTS FOR (a:Actor) REQUIRE a.name IS UNIQUE",
    "CREATE CONSTRAINT plan_name IF NOT EXISTS FOR (p:Plan) REQUIRE p.name IS UNIQUE",
    "CREATE CONSTRAINT division_name IF NOT EXISTS FOR (d:Division) REQUIRE d.name IS UNIQUE",
    "CREATE CONSTRAINT skill_name IF NOT EXISTS FOR (s:Skill) REQUIRE s.name IS UNIQUE",
]
for c in constraints:
    run(c)
print("Constraints created ✅")

Constraints created ✅


In [21]:
# Divisions (الستة من الـ README)
run("""
UNWIND ['Academy','Talents','Community','Marketplace','Solutions','Services'] AS name
MERGE (:Division {name: name})
""")

# Plans (US-HUB-012)
run("""
UNWIND ['Free','Plus','Pro','Business'] AS name
MERGE (:Plan {name: name})
""")

# Actors (بيتاخدوا عند أول action)
run("""
UNWIND ['learner','buyer','talent','vendor','creator'] AS name
MERGE (:Actor {name: name})
""")

# Roles (US-005, US-HUB-044)
run("""
UNWIND ['owner','admin','member','billing_manager','super_admin',
        'division_admin','reviewer','support_agent','finance'] AS name
MERGE (:Role {name: name})
""")

print("Reference data loaded ✅")

Reference data loaded ✅


In [22]:
run("""
// Enterprise → Company → Team
MERGE (e:Enterprise {id:'ent_1', name:'TechCorp'})
MERGE (c:Company {id:'comp_1', name:'TechCorp Egypt'})
MERGE (t:Team {id:'team_1', name:'Engineering'})
MERGE (c)-[:PART_OF]->(e)
MERGE (t)-[:PART_OF]->(c)

// Users (حساب واحد لكل شخص)
MERGE (u1:User {id:'u1', name:'Ahmed', email:'ahmed@example.com', lang:'ar'})
MERGE (u2:User {id:'u2', name:'Sara',  email:'sara@example.com',  lang:'en'})
MERGE (u3:User {id:'u3', name:'Omar',  email:'omar@example.com',  lang:'ar'})

// Roles داخل المؤسسة
WITH u1,u2,u3,e,t
MATCH (owner:Role {name:'owner'}), (admin:Role {name:'admin'}), (member:Role {name:'member'})
MERGE (u1)-[:HAS_ROLE {scope:'ent_1'}]->(owner)
MERGE (u2)-[:HAS_ROLE {scope:'ent_1'}]->(admin)
MERGE (u3)-[:HAS_ROLE {scope:'team_1'}]->(member)
MERGE (u1)-[:MEMBER_OF]->(e)
MERGE (u2)-[:MEMBER_OF]->(e)
MERGE (u3)-[:MEMBER_OF]->(t)

// Plans
WITH u1,u2,u3
MATCH (free:Plan {name:'Free'}), (pro:Plan {name:'Pro'})
MERGE (u1)-[:SUBSCRIBED_TO]->(pro)
MERGE (u2)-[:SUBSCRIBED_TO]->(pro)
MERGE (u3)-[:SUBSCRIBED_TO]->(free)

// Actors (Omar بقى learner لما عمل أول action)
WITH u1,u2,u3
MATCH (learner:Actor {name:'learner'}), (buyer:Actor {name:'buyer'})
MERGE (u3)-[:ACTS_AS {since:date('2026-11-01')}]->(learner)
MERGE (u1)-[:ACTS_AS {since:date('2026-11-02')}]->(buyer)

// Consent (US-006)
WITH u3
MATCH (d:Division {name:'Academy'})
MERGE (u3)-[:CONSENTED {purpose:'analytics', version:'v1', date:date('2026-11-01')}]->(d)
""")
print("Sample data loaded ✅")

Sample data loaded ✅


In [23]:
print(run("MATCH (n) RETURN labels(n)[0] AS type, count(*) AS total ORDER BY total DESC"))

[{'type': 'Role', 'total': 9}, {'type': 'Division', 'total': 6}, {'type': 'Actor', 'total': 5}, {'type': 'Plan', 'total': 4}, {'type': 'User', 'total': 3}, {'type': 'Enterprise', 'total': 1}, {'type': 'Company', 'total': 1}, {'type': 'Team', 'total': 1}]


In [24]:
for row in run("MATCH (n) RETURN labels(n)[0] AS type, count(*) AS total ORDER BY total DESC"):
    print(f"{row['type']:<12} {row['total']}")

Role         9
Division     6
Actor        5
Plan         4
User         3
Enterprise   1
Company      1
Team         1


In [25]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")  # بيدعم عربي وإنجليزي

chroma = chromadb.Client()
profiles = chroma.get_or_create_collection(name="user_profiles", metadata={"hnsw:space": "cosine"})
print("Vector DB ready ✅")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector DB ready ✅


In [26]:
users = [
    {"id": "u1", "name": "Ahmed", "lang": "ar",
     "bio": "صاحب شركة تقنية في مصر، بدور على مطورين Python وخبراء ذكاء اصطناعي لمشاريع الشركة"},
    {"id": "u2", "name": "Sara", "lang": "en",
     "bio": "Engineering manager who leads a team of backend developers and cares about cloud infrastructure and security"},
    {"id": "u3", "name": "Omar", "lang": "ar",
     "bio": "طالب بيتعلم تطوير الويب والـ machine learning وعايز يبقى data scientist"},
]

profiles.add(
    ids=[u["id"] for u in users],
    documents=[u["bio"] for u in users],
    embeddings=model.encode([u["bio"] for u in users]).tolist(),
    metadatas=[{"name": u["name"], "lang": u["lang"]} for u in users],
)
print("Profiles embedded ✅", profiles.count())

Profiles embedded ✅ 3


In [27]:
def search(q, n=2):
    res = profiles.query(query_embeddings=model.encode([q]).tolist(), n_results=n)
    for i, doc in enumerate(res["documents"][0]):
        meta = res["metadatas"][0][i]
        dist = res["distances"][0][i]
        print(f"{meta['name']}  (distance {dist:.2f})  →  {doc[:60]}...")

search("someone learning data science")
print()
search("مدير فريق برمجة")

Omar  (distance 0.23)  →  طالب بيتعلم تطوير الويب والـ machine learning وعايز يبقى dat...
Sara  (distance 0.73)  →  Engineering manager who leads a team of backend developers a...

Sara  (distance 0.43)  →  Engineering manager who leads a team of backend developers a...
Ahmed  (distance 0.57)  →  صاحب شركة تقنية في مصر، بدور على مطورين Python وخبراء ذكاء ا...


In [30]:
def hybrid_search(query, actor="learner", purpose="analytics", division="Academy", n=3):
    # 1) Vector: المرشحين بالمعنى
    res = profiles.query(query_embeddings=model.encode([query]).tolist(), n_results=n)
    candidate_ids = res["ids"][0]
    dist = dict(zip(candidate_ids, res["distances"][0]))

    # 2) Graph: فلترة بالعلاقات
    allowed = run("""
        MATCH (u:User)-[:ACTS_AS]->(:Actor {name:$actor}),
              (u)-[:CONSENTED {purpose:$purpose}]->(:Division {name:$division})
        WHERE u.id IN $ids
        RETURN u.id AS id, u.name AS name
    """, actor=actor, purpose=purpose, division=division, ids=candidate_ids)

    for a in allowed:
        print(f"{a['name']}  (distance {dist[a['id']]:.2f})")
    if not allowed:
        print("No match")

hybrid_search("someone learning data science")

Omar  (distance 0.23)


In [31]:
hybrid_search("business owner looking for Python developers", actor="buyer", purpose="analytics")

No match
